# Multi-Class Text Classification of arXiv CS Paper Abstracts

**Task:** Classify arXiv computer-science paper abstracts into their research sub-field (6 classes) based on the abstract text alone.

**Why this is a meaningful, non-trivial problem:** Automatic subject tagging helps arXiv, journals, and recommendation systems route papers to the right reviewers/readers, and is harder than generic topic classification because the classes (e.g. *Machine Learning* vs *Machine Learning (Statistics)* vs *Artificial Intelligence*) are semantically close and vocabulary overlaps heavily (all abstracts use words like "model", "propose", "learning").

**Classes (6, satisfies the ≥4 class requirement):**
1. Machine Learning
2. Computer Vision and Pattern Recognition
3. Computation and Language (Natural Language Processing)
4. Artificial Intelligence
5. Machine Learning (Statistics)
6. Neural and Evolutionary Computing

**Notebook structure:**
1. Data loading & inspection
2. Missing values & duplicates
3. Class selection
4. Exploratory data analysis
5. Text cleaning (with a documented comparison of 3 strategies)
6. Post-cleaning QA
7. Class imbalance analysis & handling
8. Train / validation / test split
9. Text representation (TF-IDF + Word2Vec)
10. Classical ML models (Random Forest, Logistic Regression, Naive Bayes)
11. Deep learning models (SimpleRNN, GRU, LSTM, and their bidirectional variants)
12. BERT-base fine-tuning
13. Final model comparison

> **Hardware note:** This notebook is written to run on a single RTX 3060 12GB. Batch sizes, sequence lengths, and the dataset-capping strategy in Section 7 are chosen specifically so that BERT-base fine-tuning fits comfortably in 12GB of VRAM with mixed precision (fp16) enabled.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 150)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load & Inspect the Data

Dataset: arXiv computer-science paper metadata (`scientific_dataset.csv`), containing paper `title`, `summary` (abstract), and `category` (the arXiv sub-field), among other metadata columns.

In [ ]:
df = pd.read_csv('scientific_dataset.csv')

In [ ]:
df.head(10)

In [ ]:
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")

In [ ]:
df.info()

**Check:** confirm the dtypes above look right (`summary`/`title`/`category` should be `object`; any date columns should ideally be parsed as dates, though we don't need dates for this task).

## 3. Missing Values & Duplicates

This satisfies the "identify potential issues" requirement (missing values, duplicates) before we commit to a class subset.

In [ ]:
df.isnull().sum()

**Interpretation:** if `summary` or `category` have any nulls, those rows must be dropped before modeling — a paper with no abstract or no label is unusable. Re-run this cell on the real data and add a `df = df.dropna(subset=['summary', 'category'])` line here if the count is non-zero. In the cached run this notebook was originally built from, this came back at 0 for all columns, but **verify it on your machine before trusting that** — don't assume it silently.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate titles:', df['title'].duplicated().sum())
print('Duplicate summaries:', df['summary'].duplicated().sum())

**Interpretation:** 0 fully-duplicated rows, but 115 duplicate titles and 64 duplicate summaries (out of ~287K). Duplicate *titles* with different summaries are fine (e.g. same paper, different arXiv versions) and are left alone. Duplicate *summaries* are a real leakage risk (the exact same abstract could end up in both train and test), so we deduplicate on `summary` — but we do it **after** filtering down to our 6 target classes (next section), since dropping duplicates now would needlessly discard rows belonging to classes we won't even use.

## 4. Class Distribution & Target Class Selection

The full dataset has 15 categories, several with only a few thousand samples. We restrict to the **6 largest categories** — this keeps us well above the assignment's 10,000-row minimum and 4-class minimum, while avoiding categories too small to model reliably (e.g. `Sound` at 1,733 rows).

This is a deliberate scope decision: we are trading "more classes" for "enough data per class to train 10 different model architectures properly." It should be stated explicitly in the report as a limitation — the excluded categories (Robotics, Cryptography and Security, etc.) are real, valid classes, just not included in this iteration.

In [ ]:
df['category'].value_counts().head(15)

In [ ]:
top_classes = [
    'Machine Learning',
    'Computer Vision and Pattern Recognition',
    'Computation and Language (Natural Language Processing)',
    'Artificial Intelligence',
    'Machine Learning (Statistics)',
    'Neural and Evolutionary Computing'
]

data = df[df['category'].isin(top_classes)].copy()
data = data.reset_index(drop=True)
data.shape

Even at this stage the class sizes are wildly uneven (80K vs 6.7K — a ~12:1 ratio between the largest and smallest of our 6 chosen classes). We'll quantify and address this formally in Section 7, after cleaning.

## 5. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9,5))
order = data['category'].value_counts().index
sns.countplot(data=data, y='category', order=order)
plt.title('Number of papers per category')
plt.xlabel('count')
plt.ylabel('')
plt.tight_layout()
plt.show()

**Observation:** this bar chart is the class-imbalance problem made visible — `Machine Learning` and `Computer Vision` dominate, while `Neural and Evolutionary Computing` is over 10x smaller.

In [ ]:
data['summary_word_count'].describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['summary_word_count'], bins=40, kde=True)
plt.title('Distribution of abstract word count')
plt.xlabel('word count')
plt.show()

**Observation:** abstracts are roughly 140–200 words (25th–75th percentile), with a long right tail up to ~550 words. This matters later for choosing a `max_length` for tokenization/padding — 256 tokens will comfortably cover the vast majority of abstracts without excessive padding.

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=data, x='summary_word_count', y='category', order=order)
plt.title('Abstract length by category')
plt.xlabel('word count')
plt.ylabel('')
plt.tight_layout()
plt.show()

**Observation:** check whether any one category systematically has much shorter/longer abstracts than the others — if so, the model could partly be learning "length" as a shortcut feature rather than genuine topical content. Note this either way in the report.

Now we drop duplicate abstracts **within our filtered 6-class subset** (see Section 3 for reasoning):

In [ ]:
print('duplicate summaries in the filtered data:', data['summary'].duplicated().sum())
data = data.drop_duplicates(subset='summary').reset_index(drop=True)
data.shape

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(2, 3, figsize=(16,8))
axes = axes.flatten()

for ax, cat in zip(axes, top_classes):
    text = ' '.join(data[data['category'] == cat]['summary'].sample(min(2000, (data['category']==cat).sum()), random_state=RANDOM_STATE))
    wc = WordCloud(width=500, height=350, background_color='white', stopwords=None).generate(text)
    ax.imshow(wc)
    ax.set_title(cat, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

**Observation:** expect heavy overlap in the most frequent raw words across all 6 word clouds ("model", "using", "data", "method") — this is exactly why generic stopword removal alone won't be enough to separate these classes; the *combinations* and *rarer domain terms* (e.g. "convolutional", "parser", "spiking") are what will actually carry signal for the classifier.

## 6. Text Cleaning

We build a base cleaning function, then compare **three preprocessing strategies** on a sample before committing to one for the full pipeline — this directly satisfies the "compare different preprocessing strategies" requirement.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

In [ ]:
def basic_clean(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\$.*?\$', ' ', text)      # strip inline LaTeX math ($...$)
    text = re.sub(r'\\[a-zA-Z]+', ' ', text)   # strip LaTeX commands (\alpha, \frac, ...)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)       # drop digits & punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample = data['summary'].iloc[1]
print('Prev:\n', sample[:300])
print('\nCurrent:\n', basic_clean(sample)[:300])

**Design decisions worth calling out in the report:**
- Stripping `$...$` and `\command` sequences matters specifically for arXiv abstracts, since many contain inline LaTeX math (`$O(n\log n)$`, `\emph{...}`) that would otherwise pollute the vocabulary with garbage tokens.
- Dropping all digits (`[^a-z\s]`) is a trade-off: we lose numeric information (e.g. "97% accuracy", "3D") that occasionally distinguishes CV papers (which mention "3D", "2D" a lot) from others. This is a limitation worth mentioning — an alternative would be to keep digit-containing tokens like `3d`/`2d` explicitly rather than stripping all digits blindly.

In [ ]:
def clean_no_stopwords_lemma(text):
    text = basic_clean(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

def clean_keep_stopwords_lemma(text):
    text = basic_clean(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if len(w) > 2]
    return ' '.join(tokens)

def clean_no_stopwords_stem(text):
    text = basic_clean(text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

In [ ]:
sample_df = data.sample(2000, random_state=RANDOM_STATE)

v1 = sample_df['summary'].apply(clean_no_stopwords_lemma)
v2 = sample_df['summary'].apply(clean_keep_stopwords_lemma)
v3 = sample_df['summary'].apply(clean_no_stopwords_stem)

def vocab_size(series):
    words = set()
    for t in series:
        words.update(t.split())
    return len(words)

def avg_len(series):
    return series.apply(lambda t: len(t.split())).mean()

print('no stopwords + lemmatize   -> vocab:', vocab_size(v1), ' avg words/doc:', round(avg_len(v1),1))
print('keep stopwords + lemmatize -> vocab:', vocab_size(v2), ' avg words/doc:', round(avg_len(v2),1))
print('no stopwords + stemming    -> vocab:', vocab_size(v3), ' avg words/doc:', round(avg_len(v3),1))

In [ ]:
print(sample_df['summary'].iloc[0][:250], '\n')
print('no stopwords + lemma :', v1.iloc[0][:250], '\n')
print('keep stopwords       :', v2.iloc[0][:250], '\n')
print('no stopwords + stem  :', v3.iloc[0][:250])

**Decision — use `clean_no_stopwords_lemma` for the rest of the pipeline.** Justification, comparing the three on the 2,000-doc sample:
- **Stemming** produces the smallest vocabulary (~9K vs ~13K) but mangles words into non-words (`retriev`, `augment`, `larg`) — this hurts both interpretability (word clouds, feature importance) and Word2Vec/GloVe lookups, since pretrained embeddings are keyed on real words.
- **Keeping stopwords** barely changes vocabulary size but inflates average document length by ~28% with tokens ("the", "of", "and") that carry no topical signal and just add noise for TF-IDF and dilute the averaged Word2Vec document vectors.
- **No stopwords + lemmatization** is the best balance: real dictionary words (works with pretrained/self-trained embeddings), meaningfully smaller documents, and a vocabulary size similar to the "keep stopwords" version but concentrated in content-bearing words.

This is a documented, justified preprocessing decision — exactly what 3.4 in the project spec asks for.

## 7. Apply Final Cleaning & Post-Cleaning QA

In [ ]:
data['clean_summary'] = data['summary'].apply(clean_no_stopwords_lemma)
data[['summary', 'clean_summary']].head()

In [ ]:
data['clean_word_count'] = data['clean_summary'].apply(lambda t: len(t.split()))

plt.figure(figsize=(8,5))
sns.histplot(data['clean_word_count'], bins=40, kde=True)
plt.title('Word count after cleaning')
plt.xlabel('word count')
plt.show()

data['clean_word_count'].describe()

**QA check — resolve before trusting the count below.** In an earlier run of this notebook, `clean_word_count.describe()` reported a **minimum of 0.0**, but the very next cell (checking `clean_word_count == 0`) reported **0 empty rows** — those two numbers are inconsistent; if the minimum is genuinely 0.0 across a series of this length, at least one row *must* have a zero count. This kind of contradiction almost always means the cells were run out of order (e.g. `clean_word_count` was computed on a different / stale version of `data`). The cell below recomputes the check directly and explicitly, right after `clean_word_count` is (re)assigned, so this can't happen — always re-run notebooks top-to-bottom (Restart & Run All) before trusting any printed numbers, especially ones you plan to quote in a report.

In [ ]:
# Explicit, order-independent check (don't rely on .describe() min alone)
n_empty = int((data['clean_word_count'] == 0).sum())
print('rows that became empty after cleaning:', n_empty)

data = data[data['clean_word_count'] > 0].reset_index(drop=True)
print('shape after removing empty-after-cleaning rows:', data.shape)

**Additional QA — very short abstracts.** A handful of words surviving cleaning (e.g. 3–5 tokens) is barely more informative than noise and can hurt Word2Vec/TF-IDF quality. We drop abstracts with fewer than 10 cleaned tokens as an extra quality filter (document this in the report as a deliberate outlier-removal step, distinct from the empty-row filter above).

In [ ]:
MIN_TOKENS = 10
short_count = int((data['clean_word_count'] < MIN_TOKENS).sum())
print(f'abstracts with fewer than {MIN_TOKENS} tokens after cleaning:', short_count)

data = data[data['clean_word_count'] >= MIN_TOKENS].reset_index(drop=True)
print('final shape after short-document filter:', data.shape)

## 8. Class Distribution After Cleaning & Imbalance Handling

In [ ]:
data['category'].value_counts()

**Class imbalance is severe** — roughly a 12:1 ratio between the largest class (`Machine Learning`) and the smallest (`Neural and Evolutionary Computing`). Left unaddressed, this would let a model reach a deceptively high *accuracy* just by favoring the majority classes, while doing badly on macro-averaged F1 (which weighs all classes equally) — which is exactly the metric the assignment asks us to report.

**How we address it (documented decision):**
1. **Cap the majority classes** by random subsampling (stratified, reproducible with `random_state=42`) to a maximum of `CAP` rows each. This directly shrinks the imbalance ratio and, as a side benefit, keeps the dataset small enough that BERT-base fine-tuning with 3 hyperparameter configs is actually feasible in a reasonable amount of time on a single RTX 3060.
2. We do **not** upsample/duplicate the smallest class, since duplicating raw text rows for BERT/RNN training just means the model sees identical sequences more than once (limited value, some overfitting risk). Instead, residual imbalance after capping is handled with **class weights** passed into the loss function during classical ML and neural network training (Section 10 onward) — this reweights the loss so mispredicting a minority-class sample costs more than mispredicting a majority-class one.

We keep the full cleaned (uncapped) dataset saved separately too, in case you have the compute budget to train on the full ~230K rows later.

In [ ]:
CAP = 12000  # cap per class; tune down further if training time is still too long on your GPU

parts = []
for cat, g in data.groupby('category'):
    parts.append(g.sample(n=min(len(g), CAP), random_state=RANDOM_STATE))

data_capped = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print('Before capping:')
print(data['category'].value_counts())
print('\nAfter capping (CAP =', CAP, '):')
print(data_capped['category'].value_counts())
print('\nImbalance ratio before:', round(data['category'].value_counts().max() / data['category'].value_counts().min(), 2))
print('Imbalance ratio after :', round(data_capped['category'].value_counts().max() / data_capped['category'].value_counts().min(), 2))

In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=data_capped, y='category', order=data_capped['category'].value_counts().index)
plt.title(f'Class distribution after capping at {CAP}')
plt.xlabel('count')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 9. Save Cleaned Data

In [ ]:
data.to_csv('arxiv_cleaned_full.csv', index=False)
data_capped.to_csv('arxiv_cleaned_capped.csv', index=False)
print('saved full cleaned set:', data.shape)
print('saved capped modeling set:', data_capped.shape)

## 10. Train / Validation / Test Split

All modeling from here on uses `data_capped`. We use a **stratified 70/15/15 split** (train/val/test) so that the class proportions in each split match the overall (already-capped) distribution — critical given the residual imbalance. `random_state=42` throughout makes the split fully reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data_capped, test_size=0.30, stratify=data_capped['category'], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['category'], random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('train:', train_df.shape, ' val:', val_df.shape, ' test:', test_df.shape)
print('\ntrain class distribution:\n', train_df['category'].value_counts(normalize=True).round(3))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['category'])
y_val = label_encoder.transform(val_df['category'])
y_test = label_encoder.transform(test_df['category'])

num_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))
print('num_classes:', num_classes)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weight_values = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weight_values)}
print('class weights (used to counter residual imbalance during training):')
for i, cls in enumerate(label_encoder.classes_):
    print(f'  {cls:60s} -> {class_weight_dict[i]:.3f}')

## 11. Text Representation

The assignment requires **at least two of**: TF-IDF, Word2Vec, GloVe. We use **TF-IDF** (for the classical ML models) and a **self-trained Word2Vec** (for the neural network models). GloVe is a reasonable third option if you want to compare embedding quality, but is skipped here to keep scope manageable — note this in the report as a possible extension.

Both representations are **fit only on the training set** (fit_transform on train, transform on val/test) to avoid data leakage.

### 11.1 TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_summary'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['clean_summary'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['clean_summary'])

print('TF-IDF shapes -> train:', X_train_tfidf.shape, ' val:', X_val_tfidf.shape, ' test:', X_test_tfidf.shape)

### 11.2 Word2Vec (self-trained on the training corpus)

In [ ]:
from gensim.models import Word2Vec

EMBED_DIM = 200

tokenized_train = train_df['clean_summary'].apply(str.split).tolist()

w2v_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=EMBED_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram; generally stronger than CBOW for smaller/domain-specific corpora
    epochs=10,
    seed=RANDOM_STATE,
)

print('Word2Vec vocabulary size:', len(w2v_model.wv))
print('Most similar to "neural":', w2v_model.wv.most_similar('neural', topn=5))

**Note:** we train Word2Vec on the training split's vocabulary. This means any word appearing only in val/test and not in train will have no embedding — this is expected and standard practice (it mirrors how the model would behave on genuinely unseen words in production); the Keras Tokenizer/embedding matrix step below handles out-of-vocabulary words by mapping them to a zero vector.

## 12. Classical ML Models

Three models: **Random Forest**, **Logistic Regression**, **Naive Bayes** — all trained on the TF-IDF features. For each, we manually try **3 hyperparameter configurations**, evaluate each on the validation set (macro-F1, since it reflects performance across all classes equally despite the residual imbalance), log every run in a results table, then pick the best config per model and do the final evaluation on the untouched test set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

tuning_log = []  # every (model, config, val metrics) row goes here for the report appendix

def evaluate_on_val(model, X_val, y_val):
    pred = model.predict(X_val)
    return accuracy_score(y_val, pred), f1_score(y_val, pred, average='macro')

def log_run(model_name, config, val_acc, val_f1):
    tuning_log.append({'model': model_name, 'config': str(config), 'val_accuracy': val_acc, 'val_f1_macro': val_f1})
    print(f"{model_name:22s} {str(config):55s} val_acc={val_acc:.4f}  val_f1_macro={val_f1:.4f}")

In [ ]:
# ---- Random Forest: 3 configs ----
rf_configs = [
    {'n_estimators': 100, 'max_depth': None},
    {'n_estimators': 300, 'max_depth': 40},
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2},
]

rf_models = {}
for cfg in rf_configs:
    clf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced', **cfg)
    clf.fit(X_train_tfidf, y_train)
    acc, f1m = evaluate_on_val(clf, X_val_tfidf, y_val)
    log_run('RandomForest', cfg, acc, f1m)
    rf_models[str(cfg)] = clf

In [ ]:
# ---- Logistic Regression: 3 configs ----
lr_configs = [
    {'C': 0.1, 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 5.0, 'solver': 'saga', 'max_iter': 2000},
]

lr_models = {}
for cfg in lr_configs:
    clf = LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, **cfg)
    clf.fit(X_train_tfidf, y_train)
    acc, f1m = evaluate_on_val(clf, X_val_tfidf, y_val)
    log_run('LogisticRegression', cfg, acc, f1m)
    lr_models[str(cfg)] = clf

In [ ]:
# ---- Naive Bayes: 3 configs ----
# MultinomialNB has no native class_weight param; we approximate class balancing via `sample_weight` at fit time.
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_train = compute_sample_weight('balanced', y_train)

nb_configs = [
    {'alpha': 0.1},
    {'alpha': 0.5},
    {'alpha': 1.0},
]

nb_models = {}
for cfg in nb_configs:
    clf = MultinomialNB(**cfg)
    clf.fit(X_train_tfidf, y_train, sample_weight=sample_weights_train)
    acc, f1m = evaluate_on_val(clf, X_val_tfidf, y_val)
    log_run('NaiveBayes', cfg, acc, f1m)
    nb_models[str(cfg)] = clf

In [ ]:
tuning_log_df = pd.DataFrame(tuning_log)
tuning_log_df

Pick the best configuration per model family (highest validation macro-F1), then do the one-shot final evaluation on the held-out **test** set:

In [ ]:
def best_config_row(df, model_name):
    sub = df[df['model'] == model_name]
    return sub.loc[sub['val_f1_macro'].idxmax()]

classical_models = {'RandomForest': rf_models, 'LogisticRegression': lr_models, 'NaiveBayes': nb_models}
final_results = []  # collects test-set metrics for every model in the whole notebook (Section 13 will keep appending here)

def evaluate_on_test(model_name, model, X_test, y_test, label_encoder):
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1_macro = f1_score(y_test, pred, average='macro')
    f1_weighted = f1_score(y_test, pred, average='weighted')
    cm = confusion_matrix(y_test, pred)
    report = classification_report(y_test, pred, target_names=label_encoder.classes_)

    print(f'=== {model_name} (test set) ===')
    print(f'Accuracy: {acc:.4f}   Macro-F1: {f1_macro:.4f}   Weighted-F1: {f1_weighted:.4f}')
    print(report)

    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix — {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    final_results.append({'model': model_name, 'test_accuracy': acc, 'test_f1_macro': f1_macro, 'test_f1_weighted': f1_weighted})
    return pred

for model_name, model_dict in classical_models.items():
    best_row = best_config_row(tuning_log_df, model_name)
    best_model = model_dict[best_row['config']]
    print(f'Best config for {model_name}: {best_row["config"]}  (val_f1_macro={best_row["val_f1_macro"]:.4f})')
    evaluate_on_test(model_name, best_model, X_test_tfidf, y_test, label_encoder)

## 13. Deep Learning Models — RNN Family

Six architectures required: **SimpleRNN, GRU, LSTM**, and their **Bidirectional** counterparts. All share the same data prep (Keras tokenizer + padded sequences + a Word2Vec-initialized embedding matrix) and the same training/tuning loop, parameterized by architecture.

**GPU notes for the RTX 3060 (12GB):** this whole family is lightweight — batch size 128, embedding dim 200, and up to 256 hidden units per direction all fit with room to spare. `tf.data`/Keras will use the GPU automatically if TensorFlow detects it; the explicit device check below just confirms that.

In [ ]:
import tensorflow as tf
print('GPUs visible to TensorFlow:', tf.config.list_physical_devices('GPU'))

# Optional but recommended on a 12GB card: let TF grow GPU memory allocation as needed
# instead of grabbing all 12GB up front (avoids OOM conflicts with anything else using the GPU).
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 30000
MAX_LEN = 256  # covers the large majority of cleaned abstracts (see Section 5 word-count distribution)

keras_tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tokenizer.fit_on_texts(train_df['clean_summary'])

X_train_seq = pad_sequences(keras_tokenizer.texts_to_sequences(train_df['clean_summary']), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq = pad_sequences(keras_tokenizer.texts_to_sequences(val_df['clean_summary']), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq = pad_sequences(keras_tokenizer.texts_to_sequences(test_df['clean_summary']), maxlen=MAX_LEN, padding='post', truncating='post')

vocab_size = min(MAX_VOCAB, len(keras_tokenizer.word_index) + 1)
print('Effective vocab size:', vocab_size)
print('Sequence shapes -> train:', X_train_seq.shape, ' val:', X_val_seq.shape, ' test:', X_test_seq.shape)

In [ ]:
# Build an embedding matrix from the Word2Vec model trained in Section 11.2.
# Words in the Keras vocab but not in the Word2Vec vocab (rare/OOV words) get a zero vector.
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
hits, misses = 0, 0
for word, idx in keras_tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        hits += 1
    else:
        misses += 1

print(f'Embedding matrix: {hits} words matched to Word2Vec, {misses} words unmatched (left as zero vectors)')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

def build_rnn_model(cell_type, units, bidirectional, dropout, learning_rate):
    cell_cls = {'SimpleRNN': SimpleRNN, 'GRU': GRU, 'LSTM': LSTM}[cell_type]
    recurrent_layer = cell_cls(units, dropout=dropout)
    if bidirectional:
        recurrent_layer = Bidirectional(recurrent_layer)

    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, weights=[embedding_matrix],
                  trainable=True, mask_zero=True),
        recurrent_layer,
        Dropout(dropout),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

**Architectures and hyperparameter grid.** Each of the 6 architectures gets 3 configurations (units / dropout / learning rate), trained with early stopping on validation loss (patience=2) so weaker configs don't waste GPU time. All results are logged to `tuning_log` (same table as the classical models, for one unified appendix table in the report).

In [ ]:
rnn_architectures = [
    ('SimpleRNN', False),
    ('GRU', False),
    ('LSTM', False),
    ('SimpleRNN', True),
    ('GRU', True),
    ('LSTM', True),
]

rnn_hparam_configs = [
    {'units': 64,  'dropout': 0.2, 'learning_rate': 1e-3},
    {'units': 128, 'dropout': 0.3, 'learning_rate': 1e-3},
    {'units': 128, 'dropout': 0.3, 'learning_rate': 5e-4},
]

BATCH_SIZE = 128
EPOCHS = 15

rnn_trained_models = {}

for cell_type, bidirectional in rnn_architectures:
    arch_name = ('Bi' if bidirectional else '') + cell_type
    for cfg in rnn_hparam_configs:
        tf.random.set_seed(RANDOM_STATE)
        model = build_rnn_model(cell_type, bidirectional=bidirectional, **cfg)
        es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

        history = model.fit(
            X_train_seq, y_train,
            validation_data=(X_val_seq, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weight_dict,
            callbacks=[es],
            verbose=0,
        )

        val_pred = np.argmax(model.predict(X_val_seq, batch_size=BATCH_SIZE, verbose=0), axis=1)
        val_acc = accuracy_score(y_val, val_pred)
        val_f1 = f1_score(y_val, val_pred, average='macro')
        log_run(arch_name, cfg, val_acc, val_f1)
        rnn_trained_models[(arch_name, str(cfg))] = model

In [ ]:
tuning_log_df = pd.DataFrame(tuning_log)  # refresh with the RNN runs included
tuning_log_df[tuning_log_df['model'].str.contains('RNN|GRU|LSTM')]

In [ ]:
for cell_type, bidirectional in rnn_architectures:
    arch_name = ('Bi' if bidirectional else '') + cell_type
    best_row = best_config_row(tuning_log_df, arch_name)
    best_model = rnn_trained_models[(arch_name, best_row['config'])]
    print(f'Best config for {arch_name}: {best_row["config"]}  (val_f1_macro={best_row["val_f1_macro"]:.4f})')

    val_or_test_pred = np.argmax(best_model.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0), axis=1)
    acc = accuracy_score(y_test, val_or_test_pred)
    f1_macro = f1_score(y_test, val_or_test_pred, average='macro')
    f1_weighted = f1_score(y_test, val_or_test_pred, average='weighted')
    cm = confusion_matrix(y_test, val_or_test_pred)
    report = classification_report(y_test, val_or_test_pred, target_names=label_encoder.classes_)

    print(f'=== {arch_name} (test set) ===')
    print(f'Accuracy: {acc:.4f}   Macro-F1: {f1_macro:.4f}   Weighted-F1: {f1_weighted:.4f}')
    print(report)

    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix — {arch_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    final_results.append({'model': arch_name, 'test_accuracy': acc, 'test_f1_macro': f1_macro, 'test_f1_weighted': f1_weighted})

## 14. BERT-base Fine-Tuning

**This is the most GPU-hungry part of the notebook.** On a 12GB RTX 3060:
- `max_length=256`, `batch_size=16`, and **mixed precision (fp16)** fit comfortably.
- If you hit an out-of-memory error, drop to `batch_size=8` and set `gradient_accumulation_steps=2` in `TrainingArguments` to keep the effective batch size at 16 — that halves peak memory at the cost of a bit of speed.
- Expect this section to take noticeably longer than the entire RNN section combined — 3 hyperparameter configs × several epochs each on ~66K training rows. Consider running it separately / overnight rather than as part of a single "Restart & Run All".

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

BERT_MODEL_NAME = 'bert-base-uncased'
BERT_MAX_LEN = 256

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def to_hf_dataset(df_split, labels):
    return Dataset.from_dict({'text': df_split['clean_summary'].tolist(), 'label': labels.tolist()})

# BERT is fine-tuned directly on cleaned text (not the lemmatized-and-stopword-stripped text is
# actually a reasonable choice here too, but we reuse clean_summary for consistency across models;
# note in the report that BERT's own subword tokenizer means aggressive stopword removal is less
# necessary for it than for TF-IDF/Word2Vec — an ablation comparing BERT on `summary` vs `clean_summary`
# would be a good extension if time permits).
train_ds = to_hf_dataset(train_df, y_train)
val_ds = to_hf_dataset(val_df, y_val)
test_ds = to_hf_dataset(test_df, y_test)

def tokenize_fn(batch):
    return bert_tokenizer(batch['text'], truncation=True, padding='max_length', max_length=BERT_MAX_LEN)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
from sklearn.metrics import accuracy_score as _acc, f1_score as _f1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'accuracy': _acc(labels, preds), 'f1_macro': _f1(labels, preds, average='macro')}

# 3 hyperparameter configurations, as required. Kept to 2 epochs each given how expensive BERT
# fine-tuning is relative to the rest of the pipeline -- raise `num_train_epochs` if your time
# budget allows, watching val_f1_macro for overfitting past 2-3 epochs.
bert_configs = [
    {'learning_rate': 5e-5, 'per_device_train_batch_size': 16, 'num_train_epochs': 2},
    {'learning_rate': 3e-5, 'per_device_train_batch_size': 16, 'num_train_epochs': 2},
    {'learning_rate': 2e-5, 'per_device_train_batch_size': 8,  'num_train_epochs': 3},  # smaller batch, more epochs
]

bert_trained_models = {}

for i, cfg in enumerate(bert_configs):
    print(f'--- BERT config {i+1}/{len(bert_configs)}: {cfg} ---')
    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=num_classes)

    args = TrainingArguments(
        output_dir=f'./bert_run_{i}',
        learning_rate=cfg['learning_rate'],
        per_device_train_batch_size=cfg['per_device_train_batch_size'],
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2 if cfg['per_device_train_batch_size'] == 8 else 1,  # keep effective batch ~16
        num_train_epochs=cfg['num_train_epochs'],
        eval_strategy='epoch',
        save_strategy='no',
        fp16=torch.cuda.is_available(),   # mixed precision -- essential for fitting BERT-base on 12GB
        logging_steps=100,
        seed=RANDOM_STATE,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    val_metrics = trainer.evaluate()
    log_run('BERT-base', cfg, val_metrics['eval_accuracy'], val_metrics['eval_f1_macro'])
    bert_trained_models[str(cfg)] = trainer

In [ ]:
tuning_log_df = pd.DataFrame(tuning_log)  # refresh with BERT runs included
bert_log = tuning_log_df[tuning_log_df['model'] == 'BERT-base']
bert_log

In [ ]:
best_row = best_config_row(tuning_log_df, 'BERT-base')
best_trainer = bert_trained_models[best_row['config']]
print(f'Best BERT config: {best_row["config"]}  (val_f1_macro={best_row["val_f1_macro"]:.4f})')

test_output = best_trainer.predict(test_ds)
test_pred = np.argmax(test_output.predictions, axis=1)

acc = accuracy_score(y_test, test_pred)
f1_macro = f1_score(y_test, test_pred, average='macro')
f1_weighted = f1_score(y_test, test_pred, average='weighted')
cm = confusion_matrix(y_test, test_pred)
report = classification_report(y_test, test_pred, target_names=label_encoder.classes_)

print('=== BERT-base (test set) ===')
print(f'Accuracy: {acc:.4f}   Macro-F1: {f1_macro:.4f}   Weighted-F1: {f1_weighted:.4f}')
print(report)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix — BERT-base')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

final_results.append({'model': 'BERT-base', 'test_accuracy': acc, 'test_f1_macro': f1_macro, 'test_f1_weighted': f1_weighted})

## 15. Final Model Comparison

All 10 required models (3 classical + 6 RNN-family + BERT), each evaluated at its best hyperparameter configuration, side by side on the held-out test set.

In [ ]:
final_results_df = pd.DataFrame(final_results).sort_values('test_f1_macro', ascending=False).reset_index(drop=True)
final_results_df

In [ ]:
plt.figure(figsize=(10,6))
plot_df = final_results_df.melt(id_vars='model', value_vars=['test_accuracy', 'test_f1_macro', 'test_f1_weighted'],
                                 var_name='metric', value_name='score')
sns.barplot(data=plot_df, x='score', y='model', hue='metric')
plt.title('Model comparison on the test set')
plt.xlabel('score')
plt.ylabel('')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
best_model_row = final_results_df.iloc[0]
worst_model_row = final_results_df.iloc[-1]

print(f"Best-performing model:  {best_model_row['model']}  (test macro-F1 = {best_model_row['test_f1_macro']:.4f}, accuracy = {best_model_row['test_accuracy']:.4f})")
print(f"Worst-performing model: {worst_model_row['model']}  (test macro-F1 = {worst_model_row['test_f1_macro']:.4f}, accuracy = {worst_model_row['test_accuracy']:.4f})")

In [ ]:
# Full hyperparameter tuning log (every config x every model, with validation metrics) -- for the report appendix.
tuning_log_df.sort_values(['model', 'val_f1_macro'], ascending=[True, False]).reset_index(drop=True)

### Notes for the report

- **Accuracy vs. macro-F1:** given the residual class imbalance (even after capping), always lead with macro-F1 when calling a model "best" — a model can post a high accuracy just by nailing the majority classes while quietly failing on `Neural and Evolutionary Computing`. Check the per-class confusion matrices above for exactly this pattern.
- **BERT vs. everything else:** BERT-base is expected to win on raw performance (contextual embeddings capture more nuance than averaged Word2Vec or bag-of-n-grams TF-IDF), but at dramatically higher training cost. This cost/benefit trade-off is worth a paragraph in the discussion section — is the accuracy gain worth 10-50x the training time, for this specific use case?
- **Limitations to state explicitly:** (1) only 6 of 15 arXiv categories used, (2) majority classes were capped/subsampled rather than trained on their full size, (3) GloVe was not used as a third representation, (4) digits were stripped during cleaning, losing some CV-relevant numeric tokens.
